# 리랭커 학습

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML:
        def __init__(self, data): self.data = data
        def _repr_html_(self): return self.data
    def display(value): print(value)

working_dir = Path.cwd().resolve()
candidates = [working_dir, working_dir / 'code', working_dir.parent, working_dir.parent / 'code']
CODE_ROOT = next((candidate for candidate in candidates if (candidate / 'rag').is_dir() and (candidate / 'notebooks').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('프로젝트 폴더, code/ 또는 code/notebooks/에서 실행하세요.')
sys.path.insert(0, str(CODE_ROOT))

RUN_LOCAL = False  # True로 바꾸기 전 입력 자료·모델·장치·저장 위치를 확인
print('기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.')

기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.


In [2]:
# 저장된 평가 집계와 실행 설정. 원본 값을 그대로 옮겼으며 새로 평가하지 않음.
training = {'pairs': [{'embedding': 'bge-m3', 'rows': 842, 'negative': 630, 'positive': 212},
           {'embedding': 'arctic-ko', 'rows': 823, 'negative': 602, 'positive': 221},
           {'embedding': 'kure-v1', 'rows': 826, 'negative': 613, 'positive': 213}],
 'base_model': 'BAAI/bge-reranker-v2-m3',
 'seed': 20260916,
 'folds': 5,
 'learning_rate': 2e-05,
 'epochs': 2,
 'warmup_ratio': 0.1,
 'batch_size': 4,
 'gradient_accumulation': 4,
 'max_length': 512,
 'fp16': True,
 'weight_decay': 0.01}
display(pd.DataFrame(training['pairs']).set_index('embedding'))
display(pd.Series({k:v for k,v in training.items() if k != 'pairs'}, name='setting').to_frame())

,rows,negative,positive
embedding,,,
bge-m3,842,630,212
arctic-ko,823,602,221
kure-v1,826,613,213


,setting
base_model,BAAI/bge-reranker-v2-m3
seed,20260916
folds,5
learning_rate,0.00002
epochs,2
warmup_ratio,0.1
batch_size,4
gradient_accumulation,4
max_length,512
fp16,True


## 공통 분할과 실행 범위
같은 FAQ·정답 청크·본문 해시를 공유한 문항은 하나의 그룹. 범주 층화 5겹, seed 20260916.

In [3]:
PAIRS_FILE = None   # 준비한 이진 학습쌍 JSONL: qid, chunk_id, question, text, label(0/1)
SPLIT_FILE = None   # 공통 분할 JSON: n_folds, folds[{eval_qids}], qid_to_fold
BASE_MODEL = None   # 미리 내려받은 BAAI/bge-reranker-v2-m3 로컬 폴더
OUTPUT_DIR = None   # 새 학습 결과를 저장할 비어 있는 로컬 폴더

if RUN_LOCAL:
    import os, random, hashlib
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    for value in (PAIRS_FILE, SPLIT_FILE, BASE_MODEL, OUTPUT_DIR):
        if value is None:
            raise ValueError('학습 입력·모델·출력 경로를 모두 지정하세요.')
    pairs_path, split_path, model_path, out_path = map(Path, (PAIRS_FILE, SPLIT_FILE, BASE_MODEL, OUTPUT_DIR))
    if not pairs_path.is_file() or not split_path.is_file() or not model_path.is_dir():
        raise FileNotFoundError('학습쌍·공통 분할·로컬 기본 모델을 확인하세요.')
    if out_path.exists() and any(out_path.iterdir()):
        raise FileExistsError('기존 결과를 보호하기 위해 비어 있는 출력 폴더만 허용합니다.')
    import torch, numpy as np
    from datasets import Dataset
    from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
    from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
    if not torch.cuda.is_available():
        raise RuntimeError('원래 fp16 학습 레시피 실행에는 CUDA GPU가 필요합니다.')
    rows = [json.loads(line) for line in pairs_path.read_text().splitlines() if line.strip()]
    split = json.loads(split_path.read_text())
    required = {'qid','chunk_id','question','text','label'}
    assert all(required <= row.keys() and row['label'] in (0,1) for row in rows)
    assert split['n_folds'] == 5
    out_path.mkdir(parents=True, exist_ok=True)
    seed = training['seed']
    for target in [f'fold_{k}' for k in range(5)] + ['final_all']:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        eval_ids = set() if target == 'final_all' else set(split['folds'][int(target.split('_')[1])]['eval_qids'])
        train_rows = [r for r in rows if r['qid'] not in eval_ids]
        assert not ({r['qid'] for r in train_rows} & eval_ids)
        if eval_ids:
            fold = int(target.split('_')[1])
            assert all(split['qid_to_fold'][r['qid']] != fold for r in train_rows)
        eval_positive = {r['chunk_id'] for r in rows if r['qid'] in eval_ids and r['label'] == 1}
        assert not (eval_positive & {r['chunk_id'] for r in train_rows if r['label'] == 1})
        positive = sum(r['label'] for r in train_rows)
        negative = len(train_rows) - positive
        assert positive > 0 and negative > 0
        dataset = Dataset.from_dict({'query':[r['question'] for r in train_rows], 'passage':[r['text'] for r in train_rows], 'label':[float(r['label']) for r in train_rows]})
        model = CrossEncoder(str(model_path), num_labels=1, max_length=training['max_length'])
        loss = BinaryCrossEntropyLoss(model, pos_weight=torch.tensor(negative/positive))
        target_dir = out_path / target
        args = CrossEncoderTrainingArguments(output_dir=str(out_path / (target+'_checkpoints')), num_train_epochs=training['epochs'], learning_rate=training['learning_rate'], warmup_ratio=training['warmup_ratio'], per_device_train_batch_size=training['batch_size'], gradient_accumulation_steps=training['gradient_accumulation'], weight_decay=training['weight_decay'], fp16=True, seed=seed, data_seed=seed, save_strategy='no', logging_steps=5, report_to='none')
        loss.to(args.device)
        trainer = CrossEncoderTrainer(model=model, args=args, train_dataset=dataset, loss=loss)
        trainer.train()
        model.model.half()
        model.save_pretrained(str(target_dir), safe_serialization=True)
        record = {'target':target, 'rows':len(train_rows), 'positive':positive, 'negative':negative, 'seed':seed, 'pairs_sha256':hashlib.sha256(pairs_path.read_bytes()).hexdigest(), 'split_sha256':hashlib.sha256(split_path.read_bytes()).hexdigest()}
        (target_dir / 'training_info.json').write_text(json.dumps(record, indent=2))
        del trainer, model
        torch.cuda.empty_cache()
        print(target, '학습·저장 완료')
else:
    print('학습하지 않았습니다. 저장된 학습 설정만 확인했습니다.')

학습하지 않았습니다. 저장된 학습 설정만 확인했습니다.


## 학습 후 확인
새 학습 결과는 저장된 성능표를 자동으로 대체하지 않음. 평가 fold별로 보지 않은 문항을 채점한 뒤 공용 평가 함수로 지표를 산출해야 함. 전체 학습본을 같은 질문으로 채점한 값을 OOF 결과와 혼용하지 않음. 추가 실행 의존성: datasets, accelerate. 재현 환경: sentence-transformers 5.1.2, transformers 4.57.6. 패키지 설치·GPU 실행은 자동 수행하지 않음.

## 로컬 입력 형식
학습쌍 JSONL은 실제 준비 파일의 필드(qid, chunk_id, question, text, label)를 그대로 사용하며 label은 변환된 0/1임. 공통 분할 JSON의 n_folds, folds의 eval_qids, qid_to_fold(정수 fold 번호)를 그대로 읽으므로 별도 스키마 변환은 필요 없음. 입력 파일명만 바꿔 지정 가능. 같은 FAQ·정답 청크·본문 해시의 그룹 분할을 새로 만드는 기능은 없으므로 이미 확정된 공통 분할 파일이 필요함. 원자료·학습쌍·분할 파일은 공개 저장소에 포함하지 않음.